# Big Data Processing with PySpark DataFrames

**Course alignment:** Big Data, Hadoop and Spark  
**Type:** Original supplementary practice  
**Scenario:** Aggregating a transaction stream for an e-commerce operator

This notebook introduces SparkSession, DataFrames, transformations, actions, lazy evaluation, joins, window functions and Parquet output. A local Spark session is used for learning; generating 150,000 rows locally is not evidence of cluster-scale performance.

In [ ]:
# In Google Colab or a local notebook, install PySpark if needed:
# %pip install pyspark

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("CourseBigDataPractice")
    .config("spark.sql.shuffle.partitions","8")
    .getOrCreate()
)
print("Spark version:",spark.version)
print("Master:",spark.sparkContext.master)

## 1. Create a reproducible sample dataset

In [ ]:
rng=np.random.default_rng(12)
n=150_000
start=datetime(2025,1,1)
pdf=pd.DataFrame({
    "transaction_id":np.arange(1,n+1),
    "customer_id":rng.integers(1,20_001,n),
    "region":rng.choice(["Dhaka","Chattogram","Khulna","Rajshahi"],n),
    "category":rng.choice(["Electronics","Grocery","Home","Fashion"],n),
    "quantity":rng.integers(1,5,n),
    "unit_price":rng.uniform(80,12000,n).round(2),
    "order_timestamp":[start+timedelta(minutes=int(x)) for x in rng.integers(0,60*24*90,n)]
})
pdf["revenue"]=(pdf["quantity"]*pdf["unit_price"]).round(2)
transactions=spark.createDataFrame(pdf)
transactions.printSchema()
transactions.show(5,truncate=False)

## 2. Transformations and actions

Transformations such as select, filter and groupBy describe a computation. An action such as show, count or write triggers execution. Inspect the logical/physical plan rather than assuming Spark executes each line immediately.

In [ ]:
filtered=(transactions
    .filter(F.col("revenue")>=1000)
    .withColumn("order_date",F.to_date("order_timestamp"))
    .withColumn("order_month",F.date_format("order_date","yyyy-MM"))
    .select("transaction_id","customer_id","region","category","order_month","revenue")
)
filtered.explain(mode="formatted")
print("Rows after filtering:",filtered.count())

## 3. Aggregation: revenue by region and category

In [ ]:
summary=(filtered
    .groupBy("region","category")
    .agg(
        F.count("*").alias("orders"),
        F.sum("revenue").alias("revenue"),
        F.avg("revenue").alias("average_order_value")
    )
    .orderBy(F.desc("revenue"))
)
summary.show(20,truncate=False)

## 4. Join a small customer dimension

In [ ]:
customer_pdf=pd.DataFrame({
    "customer_id":np.arange(1,20_001),
    "customer_segment":rng.choice(["New","Regular","High value"],20_000,p=[.30,.55,.15])
})
customers=spark.createDataFrame(customer_pdf)
enriched=filtered.join(customers,on="customer_id",how="left")
enriched.groupBy("customer_segment").agg(
    F.count("*").alias("orders"),
    F.sum("revenue").alias("revenue")
).orderBy(F.desc("revenue")).show()

## 5. Window function: customer cumulative spend

In [ ]:
window_spec=(Window
    .partitionBy("customer_id")
    .orderBy("order_month")
    .rowsBetween(Window.unboundedPreceding,Window.currentRow)
)
customer_monthly=(filtered
    .groupBy("customer_id","order_month")
    .agg(F.sum("revenue").alias("monthly_revenue"))
    .withColumn("cumulative_revenue",F.sum("monthly_revenue").over(window_spec))
)
customer_monthly.orderBy("customer_id","order_month").show(10)

## 6. Write and read Parquet

Parquet is a columnar file format commonly used for analytical workloads. In a distributed setting, partition strategy should be chosen from query patterns and data volume; creating too many tiny partitions is inefficient.

In [ ]:
import tempfile, os
output_path=os.path.join(tempfile.gettempdir(),"course_spark_parquet")
(customer_monthly
 .repartition(4,"order_month")
 .write.mode("overwrite")
 .partitionBy("order_month")
 .parquet(output_path)
)
restored=spark.read.parquet(output_path)
print("Parquet rows:",restored.count())
restored.printSchema()

## 7. Connect the implementation to Spark architecture

- **Driver:** coordinates the Spark application and builds execution plans.
- **Executors:** execute tasks and may cache partitions in memory.
- **Transformations:** define a directed computation plan.
- **Actions:** trigger jobs.
- **DataFrame:** schema-aware distributed table abstraction.
- **RDD:** lower-level distributed collection abstraction.

## Questions and limitations

1. Which operations in the notebook are transformations and which are actions?
2. Why does the join require a shuffle in many cases? When could a broadcast join help?
3. What is the trade-off between repartitioning by month and writing many small files?
4. How would you ingest streaming meter readings and compute five-minute load aggregates?
5. How would you use Airflow or another orchestrator to sequence ingestion, validation and reporting?

This uses a local Spark runtime and synthetic data; it does not demonstrate the performance of a production multi-node cluster.

```python
# Stop the local Spark session when finished
spark.stop()
```